# 02. CatBoost 시간 기반 교차검증

실행 결과는 `results/`에 저장됩니다.

In [ ]:
from pathlib import Path
import sys

experiment_dir = Path.cwd() / "0826" if (Path.cwd() / "0826").exists() else Path.cwd()
if str(experiment_dir) not in sys.path:
    sys.path.insert(0, str(experiment_dir))


In [ ]:
"""선수·팀 ID를 포함한 범주형 변수를 CatBoost로 직접 학습한다."""

import numpy as np
from catboost import CatBoostClassifier

from common import (
    CATBOOST_CAT_COLS,
    RESULTS_DIR,
    TARGET_COL,
    VALID_YEARS,
    Timer,
    brier_metrics,
    load_train,
    prepare_catboost_frame,
    print_metrics,
    save_json,
)


PARAMS = {
    "iterations": 500,
    "depth": 8,
    "learning_rate": 0.08,
    "loss_function": "Logloss",
    "eval_metric": "BrierScore",
    "l2_leaf_reg": 5.0,
    "random_seed": 42,
    "thread_count": -1,
    "verbose": 100,
    "allow_writing_files": False,
}


def main() -> None:
    train, features = load_train()
    cat_cols = [c for c in CATBOOST_CAT_COLS if c in features]
    x = prepare_catboost_frame(train[features], cat_cols)
    predictions, targets, years = [], [], []
    fold_results = []

    for valid_year in VALID_YEARS:
        train_mask = train["season"] < valid_year
        valid_mask = train["season"] == valid_year
        model = CatBoostClassifier(**PARAMS)
        with Timer() as timer:
            model.fit(
                x.loc[train_mask],
                train.loc[train_mask, TARGET_COL],
                cat_features=cat_cols,
                eval_set=(x.loc[valid_mask], train.loc[valid_mask, TARGET_COL]),
                early_stopping_rounds=80,
                use_best_model=True,
            )
            pred = model.predict_proba(x.loc[valid_mask])[:, 1]
        y = train.loc[valid_mask, TARGET_COL].to_numpy()
        metrics = brier_metrics(y, pred)
        metrics.update(
            {
                "valid_year": valid_year,
                "seconds": timer.seconds,
                "best_iteration": model.get_best_iteration(),
            }
        )
        fold_results.append(metrics)
        print_metrics(str(valid_year), metrics)
        predictions.append(pred.astype(np.float32))
        targets.append(y.astype(np.int8))
        years.append(np.full(len(y), valid_year, dtype=np.int16))

    all_pred = np.concatenate(predictions)
    all_y = np.concatenate(targets)
    all_year = np.concatenate(years)
    overall = brier_metrics(all_y, all_pred)
    print_metrics("OOF 전체", overall)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        RESULTS_DIR / "02_catboost_oof.npz",
        y=all_y,
        prediction=all_pred,
        year=all_year,
    )
    save_json(
        RESULTS_DIR / "02_catboost_metrics.json",
        {"model": "CatBoost categorical", "params": PARAMS, "folds": fold_results, "overall": overall},
    )
    model.save_model(RESULTS_DIR / "02_catboost_last_fold.cbm")


if __name__ == "__main__":
    main()

